# Selected models: object and background segmentation

Extracted from `Action_Inference_Experiments_Object_and_Background_Segmentation_Model_Selection.ipynb`.

The two sections explicitly marked WINNER are retained: **YOLOv8n-seg active agents (people + vehicles)** and **Background Surface Grid Tracking (Active-Agent Inverse)**. Their inference, tracking, visualizations and exports are preserved. Shared tracking helpers are extracted without initializing the other candidate models.

Run from top to bottom in the project environment. The two YouTube clips must already exist in `artifacts/raw_videos/`, as in the source notebook. The scene setup generates LunarLander and CarRacing frames; the background CarRacing experiment subsequently loads the original solved PPO replay.


In [ ]:
%pip install ultralytics "imageio[ffmpeg]" "gymnasium[box2d]" pandas matplotlib pillow opencv-python


In [ ]:
# Cell 1 - imports
import json
import os
from pathlib import Path


# Notebook path setup
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
import imageio.v2 as imageio
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from IPython.display import Image as DisplayImage, Video, display
from PIL import Image, ImageDraw


In [ ]:
# Display and detection helpers
VIDEO_DIR = PROJECT_ROOT / "generated_videos"
VIDEO_DIR.mkdir(exist_ok=True)
def save_video(frames, path, fps=20):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    frames = frames.astype(np.uint8)
    try:
        imageio.mimsave(path, frames, fps=fps)
        return path
    except Exception as exc:
        fallback = path.with_suffix(".gif")
        print(f"MP4 save skipped: {type(exc).__name__}: {exc}")
        print(f"Saving GIF fallback: {fallback}")
        imageio.mimsave(fallback, frames, duration=1 / fps)
        return fallback
def show_video(frames, name, fps=20, embed=True):
    path = save_video(frames, VIDEO_DIR / name, fps=fps)
    if path.suffix.lower() == ".gif":
        display(DisplayImage(filename=str(path)))
    else:
        display(Video(str(path), embed=embed, html_attributes="controls muted loop"))
    return path
def show_video_file(path, embed=True):
    path = Path(path)
    display(Video(str(path), embed=embed, html_attributes="controls muted loop"))
    return path
def clip_indices(num_frames, clip_frames=48, seed=0):
    if num_frames <= 0:
        return np.asarray([], dtype=int)
    clip_frames = min(int(clip_frames), int(num_frames))
    rng = np.random.default_rng(seed)
    max_start = max(0, int(num_frames) - clip_frames)
    start = int(rng.integers(0, max_start + 1)) if max_start else 0
    return np.arange(start, start + clip_frames, dtype=int)
def bbox_from_mask(mask, min_area=50):
    mask = mask.astype(bool)
    visited = np.zeros(mask.shape, dtype=bool)
    boxes = []
    height, width = mask.shape
    ys, xs = np.nonzero(mask)
    for y0, x0 in zip(ys, xs):
        if visited[y0, x0] or not mask[y0, x0]:
            continue
        stack = [(int(y0), int(x0))]
        visited[y0, x0] = True
        x_min = x_max = int(x0)
        y_min = y_max = int(y0)
        area = 0
        while stack:
            y, x = stack.pop()
            area += 1
            x_min = min(x_min, x)
            x_max = max(x_max, x)
            y_min = min(y_min, y)
            y_max = max(y_max, y)
            for ny in (y - 1, y, y + 1):
                for nx in (x - 1, x, x + 1):
                    if ny == y and nx == x:
                        continue
                    if 0 <= ny < height and 0 <= nx < width and mask[ny, nx] and not visited[ny, nx]:
                        visited[ny, nx] = True
                        stack.append((ny, nx))
        if area >= min_area:
            boxes.append((x_min, y_min, x_max + 1, y_max + 1, float(area)))
    return boxes
def draw_boxes(ax, boxes, color="lime", labels=None):
    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = box[:4]
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, linewidth=2, edgecolor=color)
        ax.add_patch(rect)
        if labels:
            ax.text(x1, max(0, y1 - 4), labels[i], color=color, fontsize=9, weight="bold")
def show_detection_frame(frames, frame_idx, boxes, title, labels=None):
    plt.figure(figsize=(8, 5))
    ax = plt.gca()
    ax.imshow(frames[frame_idx])
    draw_boxes(ax, boxes, labels=labels)
    ax.set_title(title)
    ax.axis("off")
    plt.show()
def plot_detection_metric(values, title, ylabel="objects"):
    plt.figure(figsize=(8, 3))
    plt.plot(values)
    plt.title(title)
    plt.xlabel("frame")
    plt.ylabel(ylabel)
    plt.grid(alpha=0.25)
    plt.show()


In [ ]:
# Internet video helpers for independent visual problems 4 and 5
def download_video(url, path):
    import urllib.request
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        print(f"Downloading {url}")
        urllib.request.urlretrieve(url, path)
    return path
def read_video(path, max_frames=None, stride=1, start_frame=0):
    import cv2
    cap = cv2.VideoCapture(str(path))
    if start_frame:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(start_frame))
    frames_out = []
    frame_no = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if frame_no % stride == 0:
            frames_out.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            if max_frames is not None and len(frames_out) >= max_frames:
                break
        frame_no += 1
    cap.release()
    if not frames_out:
        raise RuntimeError(f"No frames read from {path}")
    return np.stack(frames_out)
def read_video_sample(path, max_frames=120, stride=3, start_seconds=0):
    import cv2
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    cap.release()
    return read_video(path, max_frames=max_frames, stride=stride, start_frame=int(start_seconds * fps))


# Visual Problems


## 1 - Lunar Lander


In [ ]:
# LunarLander setup - use kernel: Python (Action Inference)
import sys
import gymnasium as gym
from gymnasium.envs.box2d.lunar_lander import heuristic
ACTION_NAMES = {
    0: "noop",
    1: "left_engine",
    2: "main_engine",
    3: "right_engine",
}
print(sys.executable)
print(gym.__version__)


In [ ]:
# LunarLander rollout
def rollout_lunar_lander(policy, seed=0, max_steps=300, render_mode="rgb_array"):
    env = gym.make("LunarLander-v3", render_mode=render_mode)
    obs, info = env.reset(seed=seed)
    observations, actions, rewards, frames, infos = [], [], [], [], []
    for t in range(max_steps):
        action = int(policy(env.unwrapped, obs))
        next_obs, reward, terminated, truncated, info = env.step(action)
        observations.append(obs.copy())
        actions.append(action)
        rewards.append(float(reward))
        infos.append(info)
        frames.append(env.render())
        obs = next_obs
        if terminated or truncated:
            break
    env.close()
    return {
        "observations": np.asarray(observations, dtype=np.float32),
        "actions": np.asarray(actions, dtype=np.int64),
        "rewards": np.asarray(rewards, dtype=np.float32),
        "frames": np.stack(frames),
        "infos": infos,
    }
rollout = rollout_lunar_lander(heuristic, seed=0)
frames = rollout["frames"]
actions = rollout["actions"]
observations = rollout["observations"]
print("frames:", frames.shape)
print("observations:", observations.shape)
print("actions:", actions.shape)
print("action counts:", {ACTION_NAMES[i]: int((actions == i).sum()) for i in ACTION_NAMES})


In [ ]:
frame_idx = min(10, len(frames) - 1)
plt.figure(figsize=(7, 5))
plt.imshow(frames[frame_idx])
plt.title(f"t={frame_idx}, action={actions[frame_idx]} ({ACTION_NAMES[int(actions[frame_idx])]})")
plt.axis("off")
plt.show()


In [ ]:
show_video(frames, "1_lunar_lander.mp4", fps=30)


## 2 - Car Racing


In [ ]:
# CarRacing with random driving
import gymnasium as gym
env = gym.make("CarRacing-v3", render_mode="rgb_array", continuous=False)
obs, _ = env.reset(seed=0)
frames_car = []
for t in range(300):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    frames_car.append(env.render())
    if terminated or truncated:
        break
env.close()
frames_car = np.stack(frames_car)
frames_car.shape


In [ ]:
car_frame_idx = min(40, len(frames_car) - 1)
plt.figure(figsize=(7, 5))
plt.imshow(frames_car[car_frame_idx])
plt.title(f"CarRacing frame {car_frame_idx}")
plt.axis("off")
plt.show()


In [ ]:
show_video(frames_car, "2_car_racing.mp4", fps=30)


## 3 - Recorded Traffic With People


In [ ]:
# Load recorded traffic video
import cv2
import urllib.request

traffic_path = PROJECT_ROOT / "artifacts/raw_videos/traffic.avi"
traffic_path.parent.mkdir(parents=True, exist_ok=True)

if not traffic_path.exists():
    url = "https://raw.githubusercontent.com/opencv/opencv_extra/master/testdata/cv/video/768x576.avi"
    urllib.request.urlretrieve(url, traffic_path)

cap = cv2.VideoCapture(str(traffic_path))
frames_medium = []

while True:
    ok, frame = cap.read()
    if not ok:
        break
    frames_medium.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

cap.release()
frames_medium = np.stack(frames_medium)
frames_medium.shape


In [ ]:
traffic_frame_idx = min(1, len(frames_medium) - 1)
plt.figure(figsize=(8, 5))
plt.imshow(frames_medium[traffic_frame_idx])
plt.title(f"Recorded traffic frame {traffic_frame_idx}")
plt.axis("off")
plt.show()


In [ ]:
show_video(frames_medium, "3_recorded_traffic_people.mp4", fps=20)


## 4 - Random YouTube Driving Scene


In [ ]:
# Problem 4: random 3-minute YouTube driving clip
PROBLEM_4_URL = "https://www.youtube.com/watch?v=7EovwWQIvBo"
problem_4_path = PROJECT_ROOT / "artifacts/raw_videos/problem_4_youtube_random.mp4"
if not problem_4_path.exists():
    raise FileNotFoundError(f"Missing {problem_4_path}. Download it with yt-dlp first.")
# Display the full downloaded 3-minute clip.
show_video_file(problem_4_path, embed=False)
# Use a small cached sample for plotting/models. Do not decode the full 3-minute file every run.
if "frames_problem4" not in globals():
    frames_problem4 = read_video_sample(problem_4_path, max_frames=120, stride=3, start_seconds=20)
problem4_frame_idx = min(30, len(frames_problem4) - 1)
plt.figure(figsize=(8, 5))
plt.imshow(frames_problem4[problem4_frame_idx])
plt.title(f"Problem 4 sampled frame {problem4_frame_idx}")
plt.axis("off")
plt.show()


## 5 - Hard Vehicle-Crowd Interaction


In [ ]:
# Problem 5: hardest 3-minute vehicle-crowd YouTube clip
PROBLEM_5_URL = "https://www.youtube.com/watch?v=7HaJArMDKgI"
problem_5_path = PROJECT_ROOT / "artifacts/raw_videos/problem_5_youtube_hardest.mp4"
if not problem_5_path.exists():
    raise FileNotFoundError(f"Missing {problem_5_path}. Download it with yt-dlp first.")
# Display the full downloaded 3-minute clip.
show_video_file(problem_5_path, embed=False)
# Use a small cached sample for plotting/models. Do not decode the full 3-minute file every run.
if "frames_problem5" not in globals():
    frames_problem5 = read_video_sample(problem_5_path, max_frames=120, stride=3, start_seconds=20)
problem5_frame_idx = min(40, len(frames_problem5) - 1)
plt.figure(figsize=(8, 5))
plt.imshow(frames_problem5[problem5_frame_idx])
plt.title(f"Problem 5 sampled frame {problem5_frame_idx}")
plt.axis("off")
plt.show()


In [ ]:
# Shared tracking dependencies copied from the source notebook.

def box_iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    denom = area_a + area_b - inter
    return float(inter / denom) if denom > 0 else 0.0

def mask_iou(mask_a, mask_b):
    inter = np.logical_and(mask_a, mask_b).sum()
    union = np.logical_or(mask_a, mask_b).sum()
    return float(inter / union) if union > 0 else 0.0

def segmentation_match_score(track, record, stable_blob=False):
    # Match against the latest observed mask, but keep the first blob locked for output.
    track_mask = track.get("last_mask", track["mask"])
    track_box = track.get("last_box", track["box"])
    track_centroid = track.get("last_centroid", track["centroid"])
    track_area = track.get("last_area", track["area"])
    mask_score = mask_iou(track_mask, record["mask"])
    if not stable_blob:
        return mask_score, mask_score
    box_score = box_iou(track_box, record["box"])
    tx, ty = track_centroid
    rx, ry = record["centroid"]
    diag = max(1.0, np.hypot(*record["mask"].shape))
    centroid_score = max(0.0, 1.0 - (np.hypot(tx - rx, ty - ry) / (0.12 * diag)))
    area_ratio = min(track_area, record["area"]) / max(1.0, max(track_area, record["area"]))
    combined = 0.45 * mask_score + 0.25 * box_score + 0.2 * centroid_score + 0.1 * area_ratio
    return combined, mask_score

def apply_locked_blob(record, track):
    record["observed_mask"] = record["mask"]
    record["observed_box"] = record["box"]
    record["observed_centroid"] = record["centroid"]
    record["observed_area"] = record["area"]
    record["mask"] = track["locked_mask"]
    record["box"] = track["locked_box"]
    record["centroid"] = track["locked_centroid"]
    record["area"] = track["locked_area"]
    record["blob_locked"] = True
    return record

def track_segmentation_sequence(record_sequence, iou_threshold=0.25, max_missing=2, stable_blob=False):
    if stable_blob:
        iou_threshold = min(iou_threshold, 0.12)
        max_missing = max(max_missing, 8)
    tracked_sequence = []
    active_tracks = []
    next_track_id = 0
    for frame_no, records in enumerate(record_sequence):
        old_track_count = len(active_tracks)
        frame_records = [{**record, "frame": frame_no} for record in records]
        candidates = []
        used_tracks = set()
        used_records = set()
        for track_idx, track in enumerate(active_tracks):
            if track["missing"] > max_missing:
                continue
            for record_idx, record in enumerate(frame_records):
                score, mask_score = segmentation_match_score(track, record, stable_blob=stable_blob)
                threshold = 0.35 if stable_blob else iou_threshold
                if score >= threshold:
                    candidates.append((score, mask_score, track_idx, record_idx))
        for score, mask_score, track_idx, record_idx in sorted(candidates, reverse=True):
            if track_idx in used_tracks or record_idx in used_records:
                continue
            track = active_tracks[track_idx]
            record = frame_records[record_idx]
            record["track_id"] = track["track_id"]
            record["matched_iou"] = mask_score
            record["matched_score"] = score
            if stable_blob:
                apply_locked_blob(record, track)
                track.update({
                    "last_mask": record["observed_mask"],
                    "last_box": record["observed_box"],
                    "last_centroid": record["observed_centroid"],
                    "last_area": record["observed_area"],
                    "missing": 0,
                })
            else:
                track.update({
                    "mask": record["mask"],
                    "box": record["box"],
                    "centroid": record["centroid"],
                    "area": record["area"],
                    "missing": 0,
                })
            used_tracks.add(track_idx)
            used_records.add(record_idx)
        for record_idx, record in enumerate(frame_records):
            if record_idx in used_records:
                continue
            record["track_id"] = next_track_id
            record["matched_iou"] = np.nan
            record["matched_score"] = np.nan
            record["blob_locked"] = bool(stable_blob)
            active_tracks.append({
                "track_id": next_track_id,
                "mask": record["mask"],
                "box": record["box"],
                "centroid": record["centroid"],
                "area": record["area"],
                "locked_mask": record["mask"],
                "locked_box": record["box"],
                "locked_centroid": record["centroid"],
                "locked_area": record["area"],
                "last_mask": record["mask"],
                "last_box": record["box"],
                "last_centroid": record["centroid"],
                "last_area": record["area"],
                "missing": 0,
            })
            next_track_id += 1
        for track_idx, track in enumerate(active_tracks[:old_track_count]):
            if track_idx not in used_tracks:
                track["missing"] += 1
        active_tracks = [track for track in active_tracks if track["missing"] <= max_missing]
        tracked_sequence.append(sorted(frame_records, key=lambda obj: obj["track_id"]))
    return tracked_sequence


## WINNER - Object Segmentation Blob (Active Agents Only: People + Vehicles)


In [ ]:
# Minimal standalone loader for this section.
from pathlib import Path


# Notebook path setup
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image, ImageDraw
try:
    import cv2
except Exception as exc:
    cv2 = None
    print(f"cv2 unavailable: {type(exc).__name__}: {exc}")
def load_video_frames_minimal(path, max_frames=120, stride=3, start_seconds=0):
    path = Path(path)
    if cv2 is not None:
        cap = cv2.VideoCapture(str(path))
        fps = cap.get(cv2.CAP_PROP_FPS) or 30
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(start_seconds * fps))
        out = []
        frame_no = 0
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if frame_no % stride == 0:
                out.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                if len(out) >= max_frames:
                    break
            frame_no += 1
        cap.release()
        if out:
            return np.stack(out)
    reader = imageio.get_reader(path)
    out = []
    for frame_no, frame in enumerate(reader):
        if frame_no % stride == 0:
            out.append(np.asarray(frame)[..., :3])
            if len(out) >= max_frames:
                break
    reader.close()
    if not out:
        raise RuntimeError(f"No frames loaded from {path}")
    return np.stack(out)
def ensure_frames_var(var_name, path, idx_name, idx_value, max_frames=120, stride=3, start_seconds=0):
    if var_name not in globals():
        globals()[var_name] = load_video_frames_minimal(path, max_frames=max_frames, stride=stride, start_seconds=start_seconds)
        print(f"loaded {var_name}:", globals()[var_name].shape)
    if idx_name not in globals():
        globals()[idx_name] = min(idx_value, len(globals()[var_name]) - 1)
        print(f"set {idx_name}:", globals()[idx_name])
ensure_frames_var("frames", "generated_videos/1_lunar_lander.gif", "frame_idx", 10, max_frames=80, stride=1)
ensure_frames_var("frames_car", "generated_videos/2_car_racing.gif", "car_frame_idx", 40, max_frames=120, stride=1)
ensure_frames_var("frames_medium", "artifacts/raw_videos/traffic.avi", "traffic_frame_idx", 1, max_frames=120, stride=1)
ensure_frames_var("frames_problem4", "artifacts/raw_videos/problem_4_youtube_random.mp4", "problem4_frame_idx", 30, max_frames=120, stride=3, start_seconds=20)
ensure_frames_var("frames_problem5", "artifacts/raw_videos/problem_5_youtube_hardest.mp4", "problem5_frame_idx", 40, max_frames=120, stride=3, start_seconds=20)
from ultralytics import YOLO
active_agent_seg_model = YOLO("models/yolov8n-seg.pt")
VIDEO_DIR = PROJECT_ROOT / "generated_videos"
VIDEO_DIR.mkdir(exist_ok=True)
def save_video(frames, path, fps=20):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    frames = frames.astype(np.uint8)
    try:
        imageio.mimsave(path, frames, fps=fps)
        return path
    except Exception as exc:
        fallback = path.with_suffix(".gif")
        print(f"MP4 save skipped: {type(exc).__name__}: {exc}")
        print(f"Saving GIF fallback: {fallback}")
        imageio.mimsave(fallback, frames, duration=1 / fps)
        return fallback
def show_video(frames, name, fps=20, embed=True):
    from IPython.display import Image as DisplayImage, Video
    path = save_video(frames, VIDEO_DIR / name, fps=fps)
    if path.suffix.lower() == ".gif":
        display(DisplayImage(filename=str(path)))
    else:
        display(Video(str(path), embed=embed, html_attributes="controls muted loop"))
    return path
def clip_indices(num_frames, clip_frames=48, seed=0):
    if num_frames <= 0:
        return np.asarray([], dtype=int)
    clip_frames = min(int(clip_frames), int(num_frames))
    rng = np.random.default_rng(seed)
    max_start = max(0, int(num_frames) - clip_frames)
    start = int(rng.integers(0, max_start + 1)) if max_start else 0
    return np.arange(start, start + clip_frames, dtype=int)
def mask_to_record(mask, object_id):
    ys, xs = np.nonzero(mask)
    if len(xs) == 0:
        return None
    x1, x2 = int(xs.min()), int(xs.max()) + 1
    y1, y2 = int(ys.min()), int(ys.max()) + 1
    area = int(mask.sum())
    centroid = (float(xs.mean()), float(ys.mean()))
    return {
        "object_id": object_id,
        "mask": mask,
        "box": (x1, y1, x2, y2),
        "area": area,
        "centroid": centroid,
    }
# Active-agent-only segmentation: people + vehicles.
# Background is everything outside these selected class masks.
ACTIVE_AGENT_CLASS_NAMES = {
    "person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck", "boat"
}
def active_agent_records_from_yolo_seg(frame, model, conf_min=0.25, max_objects=40):
    result = model(frame, verbose=False, conf=conf_min)[0]
    records = []
    if result.masks is None:
        return records
    masks = result.masks.data.cpu().numpy().astype(bool)
    if masks.shape[1:] != frame.shape[:2]:
        import cv2
        masks = np.asarray([
            cv2.resize(mask.astype(np.uint8), (frame.shape[1], frame.shape[0]), interpolation=cv2.INTER_NEAREST).astype(bool)
            for mask in masks
        ])
    classes = result.boxes.cls.cpu().numpy().astype(int)
    confs = result.boxes.conf.cpu().numpy()
    for mask, class_id, conf in zip(masks, classes, confs):
        class_name = result.names[int(class_id)]
        if class_name not in ACTIVE_AGENT_CLASS_NAMES or float(conf) < conf_min:
            continue
        record = mask_to_record(mask, len(records))
        if record is None:
            continue
        record["class_id"] = int(class_id)
        record["class_name"] = class_name
        record["confidence"] = float(conf)
        records.append(record)
    return sorted(records, key=lambda obj: obj["area"], reverse=True)[:max_objects]
def render_active_agents_inverse_background(frame, records, title, alpha_bg=0.35, alpha_agent=0.55):
    active_mask = np.zeros(frame.shape[:2], dtype=bool)
    for record in records:
        active_mask |= record["mask"]
    bg_mask = ~active_mask
    image = frame.copy()
    image[bg_mask] = (image[bg_mask] * (1 - alpha_bg) + np.array([40, 180, 90]) * alpha_bg).astype(np.uint8)
    image[active_mask] = (image[active_mask] * (1 - alpha_agent) + np.array([230, 50, 40]) * alpha_agent).astype(np.uint8)
    plt.figure(figsize=(8, 5))
    plt.imshow(image)
    plt.title(title)
    plt.axis("off")
    plt.show()
    return bg_mask, active_mask
def show_active_agent_only_segmentation_experiment(frames_batch, frame_idx, title, model=None, conf_min=0.25):
    if model is None:
        model = active_agent_seg_model
    frame = frames_batch[frame_idx]
    records = active_agent_records_from_yolo_seg(frame, model=model, conf_min=conf_min)
    bg_mask, active_mask = render_active_agents_inverse_background(frame, records, title)
    rows = []
    for obj in records:
        rows.append({
            "object_id": obj["object_id"],
            "class_name": obj["class_name"],
            "confidence": round(obj["confidence"], 3),
            "area": obj["area"],
            "centroid_x": round(obj["centroid"][0], 2),
            "centroid_y": round(obj["centroid"][1], 2),
            "box": obj["box"],
        })
    summary = pd.DataFrame([{
        "frame": int(frame_idx),
        "selected_classes": sorted(ACTIVE_AGENT_CLASS_NAMES),
        "agents": len(records),
        "active_agent_pixels": int(active_mask.sum()),
        "background_pixels": int(bg_mask.sum()),
        "background_ratio": float(bg_mask.mean()),
    }])
    display(summary)
    display(pd.DataFrame(rows))
    return bg_mask, records, summary
def save_active_agent_video(frames_batch, name, model=None, fps=6, clip_frames=36, seed=0, conf_min=0.25):
    if model is None:
        model = active_agent_seg_model
    (PROJECT_ROOT / "generated_videos").mkdir(exist_ok=True)
    indices = np.arange(min(len(frames_batch), clip_frames), dtype=int)
    if len(frames_batch) > clip_frames:
        indices = clip_indices(len(frames_batch), clip_frames=clip_frames, seed=seed) if "clip_indices" in globals() else indices
    rendered = []
    raw_sequence = []
    for i in indices:
        records = active_agent_records_from_yolo_seg(frames_batch[i], model=model, conf_min=conf_min)
        raw_sequence.append(records)
        active_mask = np.zeros(frames_batch[i].shape[:2], dtype=bool)
        for record in records:
            active_mask |= record["mask"]
        bg_mask = ~active_mask
        image = frames_batch[i].copy()
        image[bg_mask] = (image[bg_mask] * 0.65 + np.array([40, 180, 90]) * 0.35).astype(np.uint8)
        image[active_mask] = (image[active_mask] * 0.45 + np.array([230, 50, 40]) * 0.55).astype(np.uint8)
        rendered.append(image)
    tracked_sequence = track_segmentation_sequence(raw_sequence, stable_blob=True) if "track_segmentation_sequence" in globals() else raw_sequence
    print(f"video frames: {int(indices[0]) if len(indices) else 0}:{int(indices[-1]) + 1 if len(indices) else 0}")
    path = show_video(np.asarray(rendered), name, fps=fps) if "show_video" in globals() else None
    return path, tracked_sequence
def show_active_agent_only_segmentation_experiment(frames_batch, frame_idx, title, model=None, conf_min=0.25, video_name=None):
    if model is None:
        model = active_agent_seg_model
    frame = frames_batch[frame_idx]
    records = active_agent_records_from_yolo_seg(frame, model=model, conf_min=conf_min)
    bg_mask, active_mask = render_active_agents_inverse_background(frame, records, title)
    video_path = None
    video_tracks = None
    if video_name is not None:
        video_path, video_tracks = save_active_agent_video(frames_batch, video_name, model=model, conf_min=conf_min)
    rows = []
    for obj in records:
        rows.append({
            "object_id": obj["object_id"],
            "class_name": obj["class_name"],
            "confidence": round(obj["confidence"], 3),
            "area": obj["area"],
            "centroid_x": round(obj["centroid"][0], 2),
            "centroid_y": round(obj["centroid"][1], 2),
            "box": obj["box"],
        })
    summary = pd.DataFrame([{
        "frame": int(frame_idx),
        "selected_classes": sorted(ACTIVE_AGENT_CLASS_NAMES),
        "agents": len(records),
        "active_agent_pixels": int(active_mask.sum()),
        "background_pixels": int(bg_mask.sum()),
        "background_ratio": float(bg_mask.mean()),
        "video_path": str(video_path) if video_path is not None else None,
    }])
    display(summary)
    display(pd.DataFrame(rows))
    return bg_mask, records, summary


### 1 - Lunar Lander


In [ ]:
active_agents_bg_1_lunar_mask, active_agents_1_lunar_records, active_agents_1_lunar_data = show_active_agent_only_segmentation_experiment(
    frames,
    frame_idx,
    "1 - Lunar Lander - Active Agents Only: People + Vehicles",
    video_name="active_agents_only_1_lunar.mp4"
)


### 2 - Car Racing


In [ ]:
active_agents_bg_2_car_mask, active_agents_2_car_records, active_agents_2_car_data = show_active_agent_only_segmentation_experiment(
    frames_car,
    car_frame_idx,
    "2 - Car Racing - Active Agents Only: People + Vehicles",
    video_name="active_agents_only_2_car.mp4"
)


### 3 - Recorded Traffic With People


In [ ]:
active_agents_bg_3_traffic_mask, active_agents_3_traffic_records, active_agents_3_traffic_data = show_active_agent_only_segmentation_experiment(
    frames_medium,
    traffic_frame_idx,
    "3 - Recorded Traffic With People - Active Agents Only: People + Vehicles",
    video_name="active_agents_only_3_traffic.mp4"
)


### 4 - Random YouTube Driving Scene


In [ ]:
active_agents_bg_4_youtube_mask, active_agents_4_youtube_records, active_agents_4_youtube_data = show_active_agent_only_segmentation_experiment(
    frames_problem4,
    problem4_frame_idx,
    "4 - Random YouTube Driving Scene - Active Agents Only: People + Vehicles",
    video_name="active_agents_only_4_youtube.mp4"
)


### 5 - Hard Vehicle-Crowd Interaction


In [ ]:
active_agents_bg_5_hard_mask, active_agents_5_hard_records, active_agents_5_hard_data = show_active_agent_only_segmentation_experiment(
    frames_problem5,
    problem5_frame_idx,
    "5 - Hard Vehicle-Crowd Interaction - Active Agents Only: People + Vehicles",
    video_name="active_agents_only_5_hard.mp4"
)


# Background Segmentation


Goal: mark background pixels/regions without active agents.


## Winner - Background Surface Grid Tracking (Active-Agent Inverse)

In [ ]:
# Background surface grid tracking from Active Agents Only inverse mask.
from pathlib import Path


# Notebook path setup
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
import matplotlib.pyplot as plt
import imageio.v2 as imageio
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image
!pip install "imageio[ffmpeg]" -q

def ensure_surface_grid_dependencies():
    global active_agent_seg_model
    if "active_agent_seg_model" not in globals():
        from ultralytics import YOLO
        active_agent_seg_model = YOLO("models/yolov8n-seg.pt")
    if "ACTIVE_AGENT_CLASS_NAMES" not in globals():
        globals()["ACTIVE_AGENT_CLASS_NAMES"] = {"person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck", "boat"}
    if "mask_to_record" not in globals():
        def mask_to_record(mask, object_id):
            ys, xs = np.nonzero(mask)
            if len(xs) == 0:
                return None
            x1, x2 = int(xs.min()), int(xs.max()) + 1
            y1, y2 = int(ys.min()), int(ys.max()) + 1
            return {"object_id": object_id, "mask": mask, "box": (x1, y1, x2, y2), "area": int(mask.sum()), "centroid": (float(xs.mean()), float(ys.mean()))}
        globals()["mask_to_record"] = mask_to_record
    if "active_agent_records_from_yolo_seg" not in globals():
        def active_agent_records_from_yolo_seg(frame, model, conf_min=0.25, max_objects=40):
            result = model(frame, verbose=False, conf=conf_min)[0]
            records = []
            if result.masks is None:
                return records
            masks = result.masks.data.cpu().numpy().astype(bool)
            if masks.shape[1:] != frame.shape[:2]:
                import cv2
                masks = np.asarray([cv2.resize(mask.astype(np.uint8), (frame.shape[1], frame.shape[0]), interpolation=cv2.INTER_NEAREST).astype(bool) for mask in masks])
            classes = result.boxes.cls.cpu().numpy().astype(int)
            confs = result.boxes.conf.cpu().numpy()
            for mask, class_id, conf in zip(masks, classes, confs):
                class_name = result.names[int(class_id)]
                if class_name not in ACTIVE_AGENT_CLASS_NAMES or float(conf) < conf_min:
                    continue
                record = mask_to_record(mask, len(records))
                if record is None:
                    continue
                record["class_id"] = int(class_id)
                record["class_name"] = class_name
                record["confidence"] = float(conf)
                records.append(record)
            return sorted(records, key=lambda obj: obj["area"], reverse=True)[:max_objects]
        globals()["active_agent_records_from_yolo_seg"] = active_agent_records_from_yolo_seg
    if "clip_indices" not in globals():
        def clip_indices(num_frames, clip_frames=48, seed=0):
            if num_frames <= 0:
                return np.asarray([], dtype=int)
            clip_frames = min(int(clip_frames), int(num_frames))
            rng = np.random.default_rng(seed)
            max_start = max(0, int(num_frames) - clip_frames)
            start = int(rng.integers(0, max_start + 1)) if max_start else 0
            return np.arange(start, start + clip_frames, dtype=int)
        globals()["clip_indices"] = clip_indices
    if "show_video" not in globals():
        VIDEO_DIR = PROJECT_ROOT / "generated_videos"
        VIDEO_DIR.mkdir(exist_ok=True)
        def save_video(frames, path, fps=20):
            path = Path(path)
            path.parent.mkdir(parents=True, exist_ok=True)
            frames = frames.astype(np.uint8)
            try:
                imageio.mimsave(path, frames, fps=fps)
                return path
            except Exception as exc:
                fallback = path.with_suffix(".gif")
                print(f"MP4 save skipped: {type(exc).__name__}: {exc}")
                print(f"Saving GIF fallback: {fallback}")
                imageio.mimsave(fallback, frames, duration=1 / fps)
                return fallback
        def show_video(frames, name, fps=20, embed=True):
            from IPython.display import Image as DisplayImage, Video
            path = save_video(frames, VIDEO_DIR / name, fps=fps)
            if path.suffix.lower() == ".gif":
                display(DisplayImage(filename=str(path)))
            else:
                display(Video(str(path), embed=embed, html_attributes="controls muted loop"))
            return path
        globals()["show_video"] = show_video
    if "read_video" not in globals():
        def read_video(path, max_frames=None, stride=1, start_frame=0):
            import cv2
            cap = cv2.VideoCapture(str(path))
            if start_frame:
                cap.set(cv2.CAP_PROP_POS_FRAMES, int(start_frame))
            frames_out = []
            frame_no = 0
            while True:
                ok, frame = cap.read()
                if not ok:
                    break
                if frame_no % stride == 0:
                    frames_out.append(frame[:, :, ::-1])
                    if max_frames is not None and len(frames_out) >= max_frames:
                        break
                frame_no += 1
            cap.release()
            if not frames_out:
                raise RuntimeError(f"No frames read from {path}")
            return np.stack(frames_out)
        globals()["read_video"] = read_video
    if "read_video_sample" not in globals():
        def read_video_sample(path, max_frames=120, stride=3, start_seconds=0):
            import cv2
            cap = cv2.VideoCapture(str(path))
            fps = cap.get(cv2.CAP_PROP_FPS) or 30
            cap.release()
            return read_video(path, max_frames=max_frames, stride=stride, start_frame=int(start_seconds * fps))
        globals()["read_video_sample"] = read_video_sample
    if "frames" not in globals():
        try:
            import gymnasium as gym
            from gymnasium.envs.box2d.lunar_lander import heuristic
            env = gym.make("LunarLander-v3", render_mode="rgb_array")
            obs, _ = env.reset(seed=0)
            out = []
            for _ in range(80):
                action = int(heuristic(env.unwrapped, obs))
                obs, _, terminated, truncated, _ = env.step(action)
                out.append(env.render())
                if terminated or truncated:
                    break
            env.close()
            globals()["frames"] = np.stack(out)
            globals()["frame_idx"] = min(10, len(frames) - 1)
        except Exception as exc:
            print(f"Lunar standalone load skipped: {type(exc).__name__}: {exc}")
    if "frames_car" not in globals():
        try:
            import gymnasium as gym
            env = gym.make("CarRacing-v3", render_mode="rgb_array", continuous=False)
            obs, _ = env.reset(seed=0)
            out = []
            for _ in range(120):
                obs, _, terminated, truncated, _ = env.step(env.action_space.sample())
                out.append(env.render())
                if terminated or truncated:
                    break
            env.close()
            globals()["frames_car"] = np.stack(out)
            globals()["car_frame_idx"] = min(40, len(frames_car) - 1)
        except Exception as exc:
            print(f"Car standalone load skipped: {type(exc).__name__}: {exc}")
    if "frames_medium" not in globals() and (PROJECT_ROOT / "artifacts/raw_videos/traffic.avi").exists():
        globals()["frames_medium"] = read_video("artifacts/raw_videos/traffic.avi", max_frames=100, stride=1)
        globals()["traffic_frame_idx"] = min(1, len(frames_medium) - 1)
    if "frames_problem4" not in globals() and (PROJECT_ROOT / "artifacts/raw_videos/problem_4_youtube_random.mp4").exists():
        globals()["frames_problem4"] = read_video_sample("artifacts/raw_videos/problem_4_youtube_random.mp4", max_frames=120, stride=3, start_seconds=20)
        globals()["problem4_frame_idx"] = min(30, len(frames_problem4) - 1)
    if "frames_problem5" not in globals() and (PROJECT_ROOT / "artifacts/raw_videos/problem_5_youtube_hardest.mp4").exists():
        globals()["frames_problem5"] = read_video_sample("artifacts/raw_videos/problem_5_youtube_hardest.mp4", max_frames=120, stride=3, start_seconds=20)
        globals()["problem5_frame_idx"] = min(40, len(frames_problem5) - 1)

def surface_grid_background_mask(frame, conf_min=0.25, dilate_px=7):
    import cv2
    ensure_surface_grid_dependencies()
    records = active_agent_records_from_yolo_seg(frame, model=active_agent_seg_model, conf_min=conf_min)
    active_mask = np.zeros(frame.shape[:2], dtype=bool)
    for record in records:
        active_mask |= record["mask"]
    if dilate_px > 0 and active_mask.any():
        kernel = np.ones((dilate_px, dilate_px), dtype=np.uint8)
        active_mask = cv2.dilate(active_mask.astype(np.uint8), kernel, iterations=1).astype(bool)
    return ~active_mask, active_mask, records

def make_surface_grid_points(mask, step=32, margin=12):
    h, w = mask.shape
    pts = []
    for y in range(margin, h - margin, step):
        for x in range(margin, w - margin, step):
            if mask[y, x]:
                pts.append((float(x), float(y)))
    return np.asarray(pts, dtype=np.float32)

def add_missing_surface_grid_points(bg_mask, points, valid, step=32, min_distance_ratio=0.72):
    candidates = make_surface_grid_points(bg_mask, step=step)
    if len(candidates) == 0:
        return points.reshape(-1, 2), valid
    points = points.reshape(-1, 2)
    valid = valid.astype(bool)
    existing = points[valid] if len(points) else np.empty((0, 2), dtype=np.float32)
    new_points = []
    min_dist_sq = float(step * min_distance_ratio) ** 2
    for candidate in candidates:
        if len(existing):
            nearest_existing = np.min(np.sum((existing - candidate) ** 2, axis=1))
            if nearest_existing < min_dist_sq:
                continue
        if new_points:
            added = np.asarray(new_points, dtype=np.float32)
            nearest_added = np.min(np.sum((added - candidate) ** 2, axis=1))
            if nearest_added < min_dist_sq:
                continue
        new_points.append(candidate)
    if not new_points:
        return points, valid
    points = np.vstack([points, np.asarray(new_points, dtype=np.float32)]) if len(points) else np.asarray(new_points, dtype=np.float32)
    valid = np.concatenate([valid, np.ones(len(new_points), dtype=bool)])
    return points, valid

def track_surface_grid_points(frames_batch, indices, step=32):
    import cv2
    first = frames_batch[int(indices[0])]
    bg_mask, active_mask, records = surface_grid_background_mask(first)
    points0 = make_surface_grid_points(bg_mask, step=step)
    tracks = [{"frame": int(indices[0]), "points": points0, "valid": np.ones(len(points0), dtype=bool), "bg_mask": bg_mask, "active_mask": active_mask, "records": records}]
    prev_gray = cv2.cvtColor(first, cv2.COLOR_RGB2GRAY)
    prev_pts = points0.reshape(-1, 1, 2)
    valid = np.ones(len(points0), dtype=bool)
    for idx in indices[1:]:
        frame = frames_batch[int(idx)]
        gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
        bg_mask, active_mask, records = surface_grid_background_mask(frame)
        if len(prev_pts) == 0:
            next_pts = prev_pts.copy()
            valid = np.zeros_like(valid)
        else:
            next_pts, status, _ = cv2.calcOpticalFlowPyrLK(prev_gray, gray, prev_pts, None, winSize=(21, 21), maxLevel=3, criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))
            if next_pts is None or status is None:
                valid = np.zeros_like(valid)
                next_pts = prev_pts.copy()
            else:
                next_xy = next_pts.reshape(-1, 2)
                h, w = bg_mask.shape
                inside = (next_xy[:, 0] >= 0) & (next_xy[:, 0] < w) & (next_xy[:, 1] >= 0) & (next_xy[:, 1] < h)
                on_background = np.zeros(len(next_xy), dtype=bool)
                rounded = np.floor(next_xy[inside]).astype(int)
                rounded[:, 0] = np.clip(rounded[:, 0], 0, w - 1)
                rounded[:, 1] = np.clip(rounded[:, 1], 0, h - 1)
                on_background[inside] = bg_mask[rounded[:, 1], rounded[:, 0]]
                valid = valid & (status.reshape(-1) == 1) & inside & on_background
        next_xy, valid = add_missing_surface_grid_points(bg_mask, next_pts.reshape(-1, 2), valid, step=step)
        next_pts = next_xy.reshape(-1, 1, 2)
        tracks.append({"frame": int(idx), "points": next_pts.reshape(-1, 2), "valid": valid.copy(), "bg_mask": bg_mask, "active_mask": active_mask, "records": records})
        prev_gray = gray
        prev_pts = next_pts
    return tracks

def render_surface_grid_frame(frame, track, prev_track=None):
    import cv2
    image = frame.copy()
    bg_mask = track["bg_mask"]
    active_mask = track["active_mask"]
    image[bg_mask] = (image[bg_mask] * 0.72 + np.array([35, 175, 105]) * 0.28).astype(np.uint8)
    image[active_mask] = (image[active_mask] * 0.35 + np.array([230, 45, 35]) * 0.65).astype(np.uint8)
    pts = track["points"]
    valid = track["valid"]
    prev_pts = prev_track["points"] if prev_track is not None and len(prev_track["points"]) == len(pts) else None
    for i, (x, y) in enumerate(pts):
        if not valid[i]:
            continue
        x_i, y_i = int(round(x)), int(round(y))
        if prev_pts is not None:
            px, py = prev_pts[i]
            cv2.arrowedLine(image, (int(round(px)), int(round(py))), (x_i, y_i), (0, 230, 255), 1, tipLength=0.25)
        cv2.circle(image, (x_i, y_i), 3, (255, 235, 0), -1)
        cv2.circle(image, (x_i, y_i), 3, (0, 0, 0), 1)
    return image

def surface_grid_summary(tracks):
    rows = []
    base = np.empty((0, 2), dtype=np.float32)
    for track in tracks:
        pts = track["points"]
        if len(pts) > len(base):
            base = np.vstack([base, pts[len(base):]]) if len(base) else pts.copy()
        rows.append(surface_grid_summary_row(track, base))
    return pd.DataFrame(rows)

def surface_grid_summary_row(track, base_points):
    pts = track["points"]
    valid = track["valid"]
    disp = pts - base_points if len(pts) == len(base_points) else np.zeros_like(pts)
    speed = np.linalg.norm(disp, axis=1) if len(disp) else np.asarray([])
    return {
        "frame": int(track["frame"]),
        "agents": int(len(track["records"])),
        "grid_points_total": int(len(pts)),
        "grid_points_tracked": int(valid.sum()),
        "background_pixels": int(track["bg_mask"].sum()),
        "median_dx_from_start": float(np.median(disp[valid, 0])) if valid.any() else np.nan,
        "median_dy_from_start": float(np.median(disp[valid, 1])) if valid.any() else np.nan,
        "median_speed_from_start": float(np.median(speed[valid])) if valid.any() else np.nan,
    }

def show_surface_grid_tracking_full_video(video_path, title, video_name, grid_step=32, max_frames=None):
    import cv2
    from IPython.display import Video
    ensure_surface_grid_dependencies()
    input_path = Path(video_path)
    output_path = Path("generated_videos") / video_name
    output_path.parent.mkdir(exist_ok=True)
    cap = cv2.VideoCapture(str(input_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    ok, frame_bgr = cap.read()
    if not ok:
        cap.release()
        raise RuntimeError(f"No frames read from {input_path}")
    first = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    bg_mask, active_mask, records = surface_grid_background_mask(first)
    points0 = make_surface_grid_points(bg_mask, step=grid_step)
    track = {"frame": 0, "points": points0, "valid": np.ones(len(points0), dtype=bool), "bg_mask": bg_mask, "active_mask": active_mask, "records": records}
    base_points = points0.copy()
    rows = [surface_grid_summary_row(track, base_points)]
    rendered_preview = render_surface_grid_frame(first, track)
    writer = imageio.get_writer(output_path, fps=fps, macro_block_size=1)
    writer.append_data(rendered_preview)
    prev_gray = cv2.cvtColor(first, cv2.COLOR_RGB2GRAY)
    prev_pts = points0.reshape(-1, 1, 2)
    valid = np.ones(len(points0), dtype=bool)
    prev_track = track
    frame_no = 1
    try:
        while max_frames is None or frame_no < max_frames:
            ok, frame_bgr = cap.read()
            if not ok:
                break
            frame = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
            bg_mask, active_mask, records = surface_grid_background_mask(frame)
            if len(prev_pts) == 0:
                next_pts = prev_pts.copy()
                valid = np.zeros_like(valid)
            else:
                next_pts, status, _ = cv2.calcOpticalFlowPyrLK(prev_gray, gray, prev_pts, None, winSize=(21, 21), maxLevel=3, criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))
                if next_pts is None or status is None:
                    valid = np.zeros_like(valid)
                    next_pts = prev_pts.copy()
                else:
                    next_xy = next_pts.reshape(-1, 2)
                    h, w = bg_mask.shape
                    inside = (next_xy[:, 0] >= 0) & (next_xy[:, 0] < w) & (next_xy[:, 1] >= 0) & (next_xy[:, 1] < h)
                    on_background = np.zeros(len(next_xy), dtype=bool)
                    rounded = np.floor(next_xy[inside]).astype(int)
                    rounded[:, 0] = np.clip(rounded[:, 0], 0, w - 1)
                    rounded[:, 1] = np.clip(rounded[:, 1], 0, h - 1)
                    on_background[inside] = bg_mask[rounded[:, 1], rounded[:, 0]]
                    valid = valid & (status.reshape(-1) == 1) & inside & on_background
            next_xy, valid = add_missing_surface_grid_points(bg_mask, next_pts.reshape(-1, 2), valid, step=grid_step)
            if len(next_xy) > len(base_points):
                base_points = np.vstack([base_points, next_xy[len(base_points):]]) if len(base_points) else next_xy.copy()
            next_pts = next_xy.reshape(-1, 1, 2)
            track = {"frame": frame_no, "points": next_pts.reshape(-1, 2), "valid": valid.copy(), "bg_mask": bg_mask, "active_mask": active_mask, "records": records}
            writer.append_data(render_surface_grid_frame(frame, track, prev_track))
            rows.append(surface_grid_summary_row(track, base_points))
            prev_gray = gray
            prev_pts = next_pts
            prev_track = track
            frame_no += 1
            if frame_no % 100 == 0:
                print(f"processed {frame_no}/{total_frames or '?'} frames")
    finally:
        cap.release()
        writer.close()
    plt.figure(figsize=(8, 5))
    plt.imshow(rendered_preview)
    plt.title(title)
    plt.axis("off")
    plt.show()
    data = pd.DataFrame(rows)
    data["video_path"] = str(output_path)
    print(f"video frames: 0:{frame_no} from {input_path}")
    display(Video(str(output_path), embed=False, html_attributes="controls muted loop"))
    display(data.head(12))
    return output_path, data

def show_surface_grid_tracking_experiment(frames_batch, frame_idx, title, video_name, clip_frames=None, seed=0, grid_step=32):
    ensure_surface_grid_dependencies()
    indices = np.arange(len(frames_batch), dtype=int) if clip_frames is None else clip_indices(len(frames_batch), clip_frames=clip_frames, seed=seed)
    tracks = track_surface_grid_points(frames_batch, indices, step=grid_step)
    selected_pos = int(np.argmin(np.abs(indices - frame_idx))) if len(indices) else 0
    rendered = [render_surface_grid_frame(frames_batch[int(i)], track, tracks[pos - 1] if pos > 0 else None) for pos, (i, track) in enumerate(zip(indices, tracks))]
    plt.figure(figsize=(8, 5))
    plt.imshow(rendered[selected_pos])
    plt.title(title)
    plt.axis("off")
    plt.show()
    print(f"video frames: {int(indices[0]) if len(indices) else 0}:{int(indices[-1]) + 1 if len(indices) else 0}")
    video_path = show_video(np.asarray(rendered), video_name, fps=4)
    data = surface_grid_summary(tracks)
    data["video_path"] = str(video_path)
    display(data.head(12))
    return tracks, np.asarray(rendered), data
ensure_surface_grid_dependencies()



### 1 - Lunar Lander Surface Grid Tracking


In [ ]:
bg_surface_grid_1_lunar_tracks, bg_surface_grid_1_lunar_frames, bg_surface_grid_1_lunar_data = show_surface_grid_tracking_experiment(
    frames,
    frame_idx,
    "1 - Lunar Lander - Background Surface Grid Tracking",
    video_name="surface_grid_1_lunar.mp4",
)


### 2 - Car Racing Surface Grid Tracking


In [ ]:
from pathlib import Path


# Notebook path setup
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
import urllib.request
import cv2
import numpy as np

solved_car_path = PROJECT_ROOT / "artifacts/raw_videos/2_car_racing_solved_ppo_replay.mp4"
solved_car_path.parent.mkdir(parents=True, exist_ok=True)

solved_car_url = "https://huggingface.co/Brain33/ppo-car-racing-v3/resolve/main/replay.mp4"

if not solved_car_path.exists():
    urllib.request.urlretrieve(solved_car_url, solved_car_path)

cap = cv2.VideoCapture(str(solved_car_path))
frames_car = []

while True:
    ok, frame = cap.read()
    if not ok:
        break
    frames_car.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

cap.release()

frames_car = np.stack(frames_car).astype(np.uint8)
car_frame_idx = min(120, len(frames_car) - 1)

print("Solved CarRacing replay frames:", frames_car.shape)

bg_surface_grid_2_car_tracks, bg_surface_grid_2_car_frames, bg_surface_grid_2_car_data = show_surface_grid_tracking_experiment(
    frames_car,
    car_frame_idx,
    "2 - Car Racing - Background Surface Grid Tracking",
    video_name="surface_grid_2_car.mp4",
)

### 3 - Recorded Traffic With People Surface Grid Tracking


In [ ]:
bg_surface_grid_3_traffic_tracks, bg_surface_grid_3_traffic_frames, bg_surface_grid_3_traffic_data = show_surface_grid_tracking_experiment(
    frames_medium,
    traffic_frame_idx,
    "3 - Recorded Traffic With People - Background Surface Grid Tracking",
    video_name="surface_grid_3_traffic.mp4",
)


### 4 - Random YouTube Driving Scene Surface Grid Tracking


In [ ]:
bg_surface_grid_4_youtube_video_path, bg_surface_grid_4_youtube_data = show_surface_grid_tracking_full_video(
    PROJECT_ROOT / "artifacts/raw_videos/problem_4_youtube_random.mp4",
    "4 - Random YouTube Driving Scene - Background Surface Grid Tracking",
    video_name="surface_grid_4_youtube.mp4",
)


### 5 - Hard Vehicle-Crowd Interaction Surface Grid Tracking


In [ ]:
bg_surface_grid_5_hard_video_path, bg_surface_grid_5_hard_data = show_surface_grid_tracking_full_video(
    PROJECT_ROOT / "artifacts/raw_videos/problem_5_youtube_hardest.mp4",
    "5 - Hard Vehicle-Crowd Interaction - Background Surface Grid Tracking",
    video_name="surface_grid_5_hard.mp4",
)


## Save

In [ ]:
from pathlib import Path


# Notebook path setup
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
import imageio.v2 as imageio

table_dir = PROJECT_ROOT / "artifacts" / "research" / "segmentation_and_background"
video_dir = PROJECT_ROOT / "artifacts" / "research" / "full_masked_movies"
table_dir.mkdir(parents=True, exist_ok=True)
video_dir.mkdir(parents=True, exist_ok=True)

artifact_name = "1__segmentation_and_background__lunar_lander"

out_video = video_dir / f"{artifact_name}.mp4"
out_table = table_dir / f"{artifact_name}.csv"

imageio.mimsave(
    out_video,
    bg_surface_grid_1_lunar_frames.astype("uint8"),
    fps=4
)

bg_surface_grid_1_lunar_data.to_csv(
    out_table,
    index=False
)

print("saved video:", out_video)
print("saved table:", out_table)

In [ ]:
from pathlib import Path


# Notebook path setup
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
import imageio.v2 as imageio

table_dir = PROJECT_ROOT / "artifacts" / "research" / "segmentation_and_background"
video_dir = PROJECT_ROOT / "artifacts" / "research" / "full_masked_movies"
table_dir.mkdir(parents=True, exist_ok=True)
video_dir.mkdir(parents=True, exist_ok=True)

artifact_name = "2__segmentation_and_background__car_racing_old_before_rerun"

out_video = video_dir / f"{artifact_name}.mp4"
out_table = table_dir / f"{artifact_name}.csv"

imageio.mimsave(
    out_video,
    bg_surface_grid_2_car_frames.astype("uint8"),
    fps=4
)

bg_surface_grid_2_car_data.to_csv(
    out_table,
    index=False
)

print("saved video:", out_video)
print("saved table:", out_table)

In [ ]:
from pathlib import Path


# Notebook path setup
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
import imageio.v2 as imageio

table_dir = PROJECT_ROOT / "artifacts" / "research" / "segmentation_and_background"
video_dir = PROJECT_ROOT / "artifacts" / "research" / "full_masked_movies"
table_dir.mkdir(parents=True, exist_ok=True)
video_dir.mkdir(parents=True, exist_ok=True)

artifact_name = "3__segmentation_and_background__recorded_traffic_people"

out_video = video_dir / f"{artifact_name}.mp4"
out_table = table_dir / f"{artifact_name}.csv"

imageio.mimsave(
    out_video,
    bg_surface_grid_3_traffic_frames.astype("uint8"),
    fps=4
)

bg_surface_grid_3_traffic_data.to_csv(
    out_table,
    index=False
)

print("saved video:", out_video)
print("saved table:", out_table)

In [ ]:
from pathlib import Path


# Notebook path setup
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
import shutil

table_dir = PROJECT_ROOT / "artifacts" / "research" / "segmentation_and_background"
video_dir = PROJECT_ROOT / "artifacts" / "research" / "full_masked_movies"
table_dir.mkdir(parents=True, exist_ok=True)
video_dir.mkdir(parents=True, exist_ok=True)

artifact_name = "4__segmentation_and_background__3m_drive_suburbs"

shutil.copy2(
    Path(bg_surface_grid_4_youtube_video_path),
    video_dir / f"{artifact_name}.mp4"
)

bg_surface_grid_4_youtube_data.to_csv(
    table_dir / f"{artifact_name}.csv",
    index=False
)

print("saved video:", video_dir / f"{artifact_name}.mp4")
print("saved table:", table_dir / f"{artifact_name}.csv")

In [ ]:
from pathlib import Path


# Notebook path setup
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
import shutil

table_dir = PROJECT_ROOT / "artifacts" / "research" / "segmentation_and_background"
video_dir = PROJECT_ROOT / "artifacts" / "research" / "full_masked_movies"
table_dir.mkdir(parents=True, exist_ok=True)
video_dir.mkdir(parents=True, exist_ok=True)

artifact_name = "5__segmentation_and_background__3m_vehicle_crowd_hard"

shutil.copy2(
    Path(bg_surface_grid_5_hard_video_path),
    video_dir / f"{artifact_name}.mp4"
)

bg_surface_grid_5_hard_data.to_csv(
    table_dir / f"{artifact_name}.csv",
    index=False
)

print("saved video:", video_dir / f"{artifact_name}.mp4")
print("saved table:", table_dir / f"{artifact_name}.csv")